### Discharge capacity fade

How much capacity did each run lose, how well could that loss be forecast after cycle 80, and how did the discharge voltage curve change?

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

root = Path.cwd()
if not (root / "data" / "processed").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "advanced_degredation_modeling"))
from degredation_analysis import CUTOFF, load_results

In [2]:
results = load_results(root / "data" / "processed")

def label_runs(table):
    return table.assign(run=table["run_id"].str.removeprefix("LR1865SZ_cycles"))

cycles = label_runs(results["cycle_overview"])
forecasts = label_runs(results["capacity_forecasts"])
scores = label_runs(results["model_comparison"])
dvdq = label_runs(results["dvdq_curves"])

**1. Which cycles can we compare?** </br>
Check the first and last cycles for a recorded charge and discharge. Use cycles 2–100 for the comparison. The current columns give a quick check on whether discharge conditions were similar across the checkpoints.

In [3]:
checkpoints = cycles.loc[cycles["Cycle_Index"].isin([1, 2, 50, 100, 101])]
columns = ["run", "Cycle_Index", "has_charge", "has_discharge",
           "charge_median_current_a", "discharge_median_current_a", "discharge_capacity_ah"]
checkpoints[columns].sort_values(["run", "Cycle_Index"]).round(3)

,run,Cycle_Index,has_charge,has_discharge,charge_median_current_a,discharge_median_current_a,discharge_capacity_ah
101,201214_002_4,1,False,True,NaN,4.807,2.264
102,201214_002_4,2,True,True,4.807,4.807,1.687
150,201214_002_4,50,True,True,4.807,4.807,0.861
200,201214_002_4,100,True,True,4.807,4.807,0.453
201,201214_002_4,101,True,False,4.807,NaN,NaN
0,201217_001_2,1,False,True,NaN,7.196,2.271
1,201217_001_2,2,True,True,7.199,7.196,1.512
49,201217_001_2,50,True,True,7.196,7.196,0.534
99,201217_001_2,100,True,True,7.199,7.201,0.269
100,201217_001_2,101,True,False,7.200,NaN,NaN


Use cycle 2 as the measured starting point for each run as cycle 1 has missing charge. Percentage loss is relative to that run’s own cycle-2 capacity.

In [4]:
capacity = checkpoints.pivot(index="run", columns="Cycle_Index", values="discharge_capacity_ah")
fade = capacity[[2, 50, 100]].copy()
fade["loss_ah"] = fade[2] - fade[100]
fade["loss_pct"] = 100 * fade["loss_ah"] / fade[2]
fade.round(3)

Cycle_Index,2,50,100,loss_ah,loss_pct
run,,,,,
201214_002_4,1.687,0.861,0.453,1.234,73.148
201217_001_2,1.512,0.534,0.269,1.243,82.209


The earlier measurements show why both units matter. The first run fell from 1.687 to 0.453 Ah and the second from 1.512 to 0.269 Ah. Their absolute losses are similar, but the second run lost a larger fraction of its starting capacity.

**2. What can we forecast at cycle 80?** </br>
Fit each run separately on cycles 2–80 and evaluate once on cycles 81–100. With one observation per cycle, the expected counts are 79 training and 20 test measurements per run.

In [5]:
observed = cycles.loc[cycles["Cycle_Index"].between(2, 100)].copy()
observed["series"] = "Training measurements"
observed.loc[observed["Cycle_Index"] > CUTOFF, "series"] = "Held-out measurements"
observed.groupby(["run", "series"])["Cycle_Index"].agg(["min", "max", "count"])

min  max  count
run          series                                
201214_002_4 Held-out measurements   81  100     20
             Training measurements    2   80     79
201217_001_2 Held-out measurements   81  100     20
             Training measurements    2   80     79

**Capacity - fade modeling approaches:**
- The linear baseline assumes a constant Ah loss per cycle. </br>
- Exponential fade assumes a constant fractional loss. </br>
- The square-root-time model uses cycle-end times and estimates future timing from the last ten training intervals. </br>
- The LSTM uses five previous capacities and feeds each prediction into its next forecast.

In [6]:
results["model_parameters"].set_index("run_id").round(5)

,linear_intercept_ah,linear_slope_ah_per_cycle,exponential_start_ah,exponential_decay_per_cycle,sqrt_time_start_ah,sqrt_time_fade_ah_per_sqrt_h,forecast_hours_per_cycle
run_id,,,,,,,
LR1865SZ_cycles201214_002_4,1.65524,-0.01545,1.77164,0.01476,1.95213,0.23224,0.41403
LR1865SZ_cycles201217_001_2,1.27096,-0.01390,1.45902,0.02068,1.59416,0.29142,0.26903


The fitted starting capacities are estimates at cycle 2. Compare them with the measured cycle-2 values above. The exponential rate is per cycle, while the square-root-time coefficient is in Ah/√h.

In [7]:
actual = observed.rename(columns={"discharge_capacity_ah": "capacity_ah"})
predicted = forecasts.rename(columns={"model": "series", "predicted_ah": "capacity_ah"})
plot_columns = ["run", "Cycle_Index", "series", "capacity_ah"]
forecast_plot = pd.concat([actual[plot_columns], predicted[plot_columns]], ignore_index=True)
forecast_plot = forecast_plot.sort_values("Cycle_Index")

In [8]:
colors = {"Training measurements": "#94A3B8", "Held-out measurements": "#111827",
          "Linear baseline": "#64748B", "Exponential": "#2563EB",
          "Square-root time": "#B45309", "LSTM": "#0F766E"}
fig = px.line(forecast_plot, x="Cycle_Index", y="capacity_ah", color="series", facet_col="run",
              color_discrete_map=colors, labels={"Cycle_Index": "Cycle count", "capacity_ah": "Discharge capacity (Ah)"},
              title="Capacity vs. cycle count with predictive curves")
for trace in fig.data:
    if trace.name in ("Training measurements", "Held-out measurements"):
        trace.update(mode="markers", marker_size=4)
fig.add_vline(x=CUTOFF + 0.5, line_dash="dash", line_color="gray")
fig.show()

**3. Which model has the smallest forecast error?** </br>
MAE gives the average absolute miss in Ah. RMSE gives more weight to large misses. Bias is predicted minus measured capacity, so positive bias means the forecasts were high on average.

In [9]:
metrics = ["test_mae_ah", "test_rmse_ah", "test_bias_ah"]
ranking = scores.sort_values(["run", "test_mae_ah"])
ranking[["run", "approach", "model", *metrics]].round(4)

,run,approach,model,test_mae_ah,test_rmse_ah,test_bias_ah
1,201214_002_4,Statistical,Exponential,0.0218,0.0245,-0.0218
2,201214_002_4,Physics-informed,Square-root time,0.0453,0.0461,0.0453
3,201214_002_4,Machine learning,LSTM (8 units),0.0810,0.0878,0.0810
0,201214_002_4,Statistical,Linear baseline,0.2154,0.2234,-0.2154
7,201217_001_2,Machine learning,LSTM (8 units),0.0558,0.0605,0.0558
5,201217_001_2,Statistical,Exponential,0.0755,0.0761,-0.0755
6,201217_001_2,Physics-informed,Square-root time,0.0778,0.0806,-0.0778
4,201217_001_2,Statistical,Linear baseline,0.2706,0.2775,-0.2706


In [10]:
errors = scores.melt(id_vars=["run", "model"], value_vars=metrics[:2],
                     var_name="metric", value_name="error_ah")
errors["metric"] = errors["metric"].map({"test_mae_ah": "MAE", "test_rmse_ah": "RMSE"})
fig = px.bar(errors, x="run", y="error_ah", color="model", facet_col="metric",
             barmode="group", color_discrete_map=colors, labels={"error_ah": "Error (Ah)"},
             title="Discharge capacity forecast errors · Cycles 81–100")
fig.update_layout(bargap=0.18, bargroupgap=0.06)
fig.show()

**4. Where does the discharge voltage curve steepen?** </br>
Compare cycles 2, 50 and 100 within each run’s shared discharged-capacity range. The script interpolates voltage every 0.002 Ah, estimates dV/dQ with 11- and 21-point Savitzky–Golay windows, and trims the edges. More negative dV/dQ means voltage falls faster per Ah delivered.

**Voltage versus discharged capacity** </br>
Each line follows one measured discharge from start to finish. The shaded interval is the capacity range covered by all three cycles in that run. Compare voltages at the same Ah within this interval, then use the dV/dQ plot below to compare their slopes.

In [11]:
columns = ["run_id", "Cycle_Index", "Data_Point", "phase", "Capacity(Ah)", "Voltage(V)"]
path = root / "data" / "processed" / "measurements_clean.csv"
records = pd.read_csv(path, usecols=columns)
selected = records["phase"].eq("discharge") & records["Cycle_Index"].isin([2, 50, 100])
discharges = label_runs(records.loc[selected]).sort_values(["run", "Cycle_Index", "Data_Point"])
discharges["cycle"] = discharges["Cycle_Index"].astype(str)
del records

In [12]:
ranges = discharges.groupby(["run", "cycle"])["Capacity(Ah)"].agg(["min", "max"])
shared = ranges.groupby(level="run").agg(start_ah=("min", "max"), end_ah=("max", "min"))
run_order = shared.index.tolist()
cycle_colors = {"2": "#1f77b4", "50": "#ff7f0e", "100": "#2ca02c"}

In [13]:
fig = px.line(discharges, x="Capacity(Ah)", y="Voltage(V)", color="cycle", facet_col="run",
              category_orders={"run": run_order, "cycle": ["2", "50", "100"]},
              color_discrete_map=cycle_colors, labels={"Capacity(Ah)": "Discharged capacity (Ah)",
              "Voltage(V)": "Voltage (V)", "cycle": "Cycle"}, title="Discharge voltage curves")
for col, bounds in enumerate(shared.itertuples(), start=1):
    fig.add_vrect(x0=bounds.start_ah, x1=bounds.end_ah, fillcolor="gray", opacity=0.12,
                  line_width=0, layer="below", row=1, col=col)
fig.update_traces(hovertemplate="Capacity: %{x:.3f} Ah<br>Voltage: %{y:.4f} V")
fig.update_xaxes(dtick=0.25)
fig.update_layout(height=430, hovermode="closest")
fig.show()

In [14]:
slopes = dvdq.melt(id_vars=["run", "cycle", "capacity_ah"],
                   value_vars=["dvdq_11pt", "dvdq_21pt"], var_name="window", value_name="slope")
slopes["cycle"] = slopes["cycle"].astype(str)
fig = px.line(slopes, x="capacity_ah", y="slope", color="cycle", line_dash="window", facet_col="run",
              labels={"capacity_ah": "Discharged capacity (Ah)", "slope": "dV/dQ (V/Ah)"},
              title="Differential voltage across three discharge cycles")
fig.update_xaxes(tick0=0, dtick=0.25)
fig.show()

In [15]:
interval = dvdq.loc[dvdq["capacity_ah"].between(0.10, 0.20)]
median_slopes = interval.groupby(["run", "cycle"])[["dvdq_11pt", "dvdq_21pt"]].median()
median_slopes.round(3)

dvdq_11pt  dvdq_21pt
run          cycle                      
201214_002_4 2         -0.400     -0.418
             50        -0.331     -0.368
             100       -0.746     -0.732
201217_001_2 2         -0.425     -0.424
             50        -0.547     -0.574
             100       -1.632     -1.668

In the 0.10 - 0.20 Ah interval, both smoothing windows show a steeper median slope at cycle 100 than at cycle 2 in both runs. The interval represents different discharge stages as the cell fades. For the second run, 0.20 Ah is about 13% of the cycle - 2 capacity and 74% of the cycle-100 capacity. The curves support a change in full-cell voltage response. Attribution to a particular electrode would require additional evidence.

Differential voltage analysis showed that the full-cell discharge curve steepened by cycle 100 in both runs over the shared 0.10–0.20 Ah interval. The result held with both Savitzky–Golay smoothing windows. At the same discharged capacity, cycle 100 was further through its recorded discharge and closer to cutoff. The observed steepening is a candidate degradation signature in the full-cell voltage response. Assigning it to a specific electrode or degradation mechanism would require additional evidence. like


I’d first repeat the discharge tests at the same low current and controlled temperature to see whether the \(dV/dQ\) features persist. To identify which electrode contributes to the change, I’d compare the full-cell curves with reference voltage curves for the positive and negative electrodes. To confirm a specific degradation mechanism, I’d look for independent evidence, such as fresh-versus-aged electrode capacity measurements to test for loss of active material